# Prediksi Kepuasan Penumpang Maskapai Menggunakan Random Forest
**UTS Pembelajaran Mesin – Semester 5**

- **Dataset:** Airline Passenger Satisfaction (Kaggle) – 129.880 baris, 24 kolom
- **Model:** Random Forest Classifier
- **Target:** `Satisfaction` (Satisfied / Neutral or Dissatisfied)

Alur notebook: muat data → eksplorasi → preprocessing → latih model → evaluasi → feature importance → simpan model untuk web app.

## 1. Import Library

In [ ]:
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, roc_curve, confusion_matrix,
                             ConfusionMatrixDisplay, classification_report)

print("Versi scikit-learn:", sklearn.__version__)

## 2. Muat Dataset
Unduh dataset dari Kaggle (`airline_passenger_satisfaction.csv`), lalu unggah ke Colab lewat sel di bawah.

In [ ]:
from google.colab import files
uploaded = files.upload()          # pilih file airline_passenger_satisfaction.csv
csv_name = list(uploaded.keys())[0]

df = pd.read_csv(csv_name)
print("Ukuran data:", df.shape)
df.head()

## 3. Eksplorasi Data (EDA)

In [ ]:
df.info()

In [ ]:
print("Nilai kosong per kolom:")
print(df.isnull().sum()[df.isnull().sum() > 0])
print("\nData duplikat:", df.duplicated().sum())
df.describe().T

In [ ]:
# Distribusi target
counts = df["Satisfaction"].value_counts()
print(counts)
print((counts / len(df) * 100).round(1))

fig, ax = plt.subplots(figsize=(5, 3.5))
ax.bar(counts.index, counts.values, color=["#8a8a8a", "#c9a24b"])
ax.set_title("Distribusi Kepuasan Penumpang")
ax.set_ylabel("Jumlah penumpang")
plt.xticks(rotation=10)
plt.show()

In [ ]:
# Kepuasan berdasarkan kelas dan tipe perjalanan
for col in ["Class", "Type of Travel", "Customer Type"]:
    tab = pd.crosstab(df[col], df["Satisfaction"], normalize="index") * 100
    print(f"\nPersentase kepuasan menurut {col}:")
    print(tab.round(1))

## 4. Preprocessing
1. Buang kolom `ID` (hanya pengenal, tidak berguna untuk model).
2. Isi nilai kosong pada `Arrival Delay` dengan median.
3. Ubah kolom kategorikal menjadi angka.
4. Ubah target menjadi 0 (Neutral or Dissatisfied) dan 1 (Satisfied).

In [ ]:
MAP_GENDER   = {"Female": 0, "Male": 1}
MAP_CUSTOMER = {"First-time": 0, "Returning": 1}
MAP_TRAVEL   = {"Personal": 0, "Business": 1}
MAP_CLASS    = {"Economy": 0, "Economy Plus": 1, "Business": 2}
TARGET_MAP   = {"Neutral or Dissatisfied": 0, "Satisfied": 1}

data = df.drop(columns=["ID"]).copy()
data["Arrival Delay"] = data["Arrival Delay"].fillna(data["Arrival Delay"].median())
data["Gender"]         = data["Gender"].map(MAP_GENDER)
data["Customer Type"]  = data["Customer Type"].map(MAP_CUSTOMER)
data["Type of Travel"] = data["Type of Travel"].map(MAP_TRAVEL)
data["Class"]          = data["Class"].map(MAP_CLASS)

y = data.pop("Satisfaction").map(TARGET_MAP)
X = data

print("Jumlah fitur:", X.shape[1])
print("Nilai kosong tersisa:", int(X.isnull().sum().sum()))
X.head()

## 5. Bagi Data Latih dan Data Uji (80 : 20)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print("Data latih:", X_train.shape, "| Data uji:", X_test.shape)

## 6. Latih Model Random Forest
Parameter yang dipakai: 100 pohon, kedalaman maksimum 18, minimal 3 sampel per daun.
Pembatasan kedalaman menjaga ukuran model tetap kecil agar mudah di-deploy.

In [ ]:
model = RandomForestClassifier(
    n_estimators=100,
    max_depth=18,
    min_samples_leaf=3,
    n_jobs=-1,
    random_state=42,
)
model.fit(X_train, y_train)
print("Model selesai dilatih.")

## 7. Evaluasi Model

In [ ]:
pred  = model.predict(X_test)
proba = model.predict_proba(X_test)[:, 1]

print("Accuracy :", round(accuracy_score(y_test, pred), 4))
print("Precision:", round(precision_score(y_test, pred), 4))
print("Recall   :", round(recall_score(y_test, pred), 4))
print("F1-score :", round(f1_score(y_test, pred), 4))
print("ROC-AUC  :", round(roc_auc_score(y_test, proba), 4))
print()
print(classification_report(y_test, pred, target_names=["Neutral or Dissatisfied", "Satisfied"]))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

ConfusionMatrixDisplay.from_predictions(
    y_test, pred, display_labels=["Neutral/Dissatisfied", "Satisfied"],
    cmap="Blues", ax=axes[0], colorbar=False)
axes[0].set_title("Confusion Matrix")

fpr, tpr, _ = roc_curve(y_test, proba)
axes[1].plot(fpr, tpr, color="#c9a24b", lw=2, label=f"AUC = {roc_auc_score(y_test, proba):.3f}")
axes[1].plot([0, 1], [0, 1], "--", color="grey")
axes[1].set_xlabel("False Positive Rate")
axes[1].set_ylabel("True Positive Rate")
axes[1].set_title("Kurva ROC")
axes[1].legend()

plt.tight_layout()
plt.show()

## 8. Feature Importance

In [ ]:
importance = pd.Series(model.feature_importances_, index=X.columns).sort_values(ascending=False)
print(importance.round(4))

top = importance.head(12)[::-1]
fig, ax = plt.subplots(figsize=(7, 5))
ax.barh(top.index, top.values, color="#c9a24b")
ax.set_title("12 Fitur Paling Berpengaruh terhadap Kepuasan")
ax.set_xlabel("Tingkat kepentingan")
plt.tight_layout()
plt.show()

## 9. Uji Beberapa Kedalaman Pohon (opsional)
Perbandingan singkat untuk melihat pengaruh `max_depth` terhadap akurasi.

In [ ]:
hasil = []
for depth in [6, 10, 14, 18, None]:
    m = RandomForestClassifier(n_estimators=100, max_depth=depth, min_samples_leaf=3,
                               n_jobs=-1, random_state=42)
    m.fit(X_train, y_train)
    hasil.append({"max_depth": depth if depth else "None",
                  "akurasi_latih": round(m.score(X_train, y_train), 4),
                  "akurasi_uji": round(m.score(X_test, y_test), 4)})
pd.DataFrame(hasil)

## 10. Coba Prediksi Satu Penumpang

In [ ]:
contoh = X_test.iloc[[0]]
print(contoh.T)
label = {0: "Neutral or Dissatisfied", 1: "Satisfied"}
print("\nPrediksi:", label[int(model.predict(contoh)[0])])
print("Peluang Satisfied:", round(float(model.predict_proba(contoh)[0, 1]), 3))
print("Label sebenarnya :", label[int(y_test.iloc[0])])

## 11. Simpan Model untuk Web App
Sel ini menyimpan tiga berkas (`model.joblib`, `metrics.json`, `feature_importance.json`) dan mengunduhnya sebagai zip.
Letakkan isinya di folder `02_webapp/model/` pada proyek web.

In [ ]:
import os, shutil
os.makedirs("model", exist_ok=True)

tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()
metrics = {
    "accuracy": round(accuracy_score(y_test, pred), 4),
    "precision": round(precision_score(y_test, pred), 4),
    "recall": round(recall_score(y_test, pred), 4),
    "f1": round(f1_score(y_test, pred), 4),
    "roc_auc": round(roc_auc_score(y_test, proba), 4),
    "confusion_matrix": {"tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp)},
    "n_rows": int(len(df)),
    "n_train": int(len(X_train)),
    "n_test": int(len(X_test)),
    "n_features": int(X.shape[1]),
    "sklearn_version": sklearn.__version__,
}
imp_list = [{"feature": f, "importance": round(float(v), 4)} for f, v in importance.items()]

joblib.dump({"model": model, "features": list(X.columns)}, "model/model.joblib", compress=3)
with open("model/metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)
with open("model/feature_importance.json", "w") as f:
    json.dump(imp_list, f, indent=2)

shutil.make_archive("model_kepuasan_penumpang", "zip", "model")
print("Versi scikit-learn di Colab:", sklearn.__version__)
print("Samakan baris scikit-learn di requirements.txt web app dengan versi ini.")

from google.colab import files
files.download("model_kepuasan_penumpang.zip")